# 김민준 RPS — 실제 웹캠 데이터로 신경망 분류기 학습
기존 MediaPipe TFLite 손 검출·관절 모델은 유지하고, **새로운 4분류 신경망**을 학습합니다.
기존 Google 모델 가중치를 미세조정하는 작업은 아닙니다.
입력은 world landmark 거리 210개 + 관절 특징 12개, 출력은 rock/paper/scissors/other입니다.
CPU 런타임도 가능합니다. Colab 기본 TensorFlow를 사용하며 이 노트북에는 MediaPipe 설치가 필요 없습니다.

수집 화면에서 1회차(학습), 2회차(조정), 3회차(최종 평가)를 각각 새로 촬영하세요.
각 회차는 왼손·오른손 × 4동작으로 8회, 총 24회입니다. 각 촬영은 준비 5초 + 수집 20초입니다.
비슷한 연속 프레임을 무작위로 학습/평가에 섞지 않습니다.
최종 평가는 한 번 확인하며, 그 결과로 반복 튜닝했다면 새 최종 평가 촬영이 필요합니다.


In [ ]:
import io, json, zipfile, hashlib
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import files
from sklearn.metrics import classification_report, confusion_matrix, f1_score
tf.keras.utils.set_random_seed(42)
print('TensorFlow', tf.__version__)
print('rps_dataset.zip을 선택하세요. 관절 데이터와 손 주변 검토 사진이 Colab에 업로드됩니다.')
uploaded = files.upload()
assert len(uploaded)==1, '데이터 ZIP 하나를 선택하세요'
archive = zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values()))))
info=json.loads(archive.read('dataset.json'))
assert info['feature_version']=='world_distances_angles_v1'
labels=['rock','paper','scissors','other']
clips=[]
for name in archive.namelist():
    if name.endswith('/meta.json'):
        meta=json.loads(archive.read(name))
        rows=[json.loads(line) for line in archive.read(name.replace('meta.json','samples.jsonl')).decode().splitlines() if line]
        assert len(rows)==meta['count']
        assert meta['feature_count']==222 and meta['feature_version']==info['feature_version']
        clips.append((meta,rows))
print('촬영 수:', len(clips))
for meta,rows in clips:
    print(meta['id'], meta['split'], meta['hand'], meta['label'], len(rows), '건너뜀',meta['rejected'])


## 사진과 관절선 검토
각 촬영의 처음·중간·끝 사진을 표시합니다. 사진 한 장의 왼쪽은 원본, 오른쪽은 관절선입니다.
선이 실제 손가락을 따라가지 않거나 지정한 동작과 다르면 해당 촬영을 제외하고 Pi에서 다시 촬영하세요.
관절 자체가 틀린 사진이 많으면 이 분류기 학습보다 검출/촬영 조건 또는 영상 분류 방식 개선이 먼저입니다.


In [ ]:
from PIL import Image
for meta,rows in clips:
    images=meta.get('review_images',[])
    if not images:
        print('검토 사진 없음:',meta['id'])
        continue
    chosen=[images[i] for i in sorted(set([0,len(images)//2,len(images)-1]))]
    fig,axs=plt.subplots(1,len(chosen),figsize=(15,4),squeeze=False)
    for ax,item in zip(axs[0],chosen):
        ax.imshow(Image.open(io.BytesIO(archive.read(meta['id']+'/'+item['file']))))
        ax.axis('off')
    fig.suptitle(f"{meta['id']} / {meta['split']} / {meta['hand']} / {meta['label']}")
    plt.show()


## 데이터 구성
잘못 촬영한 ID가 있으면 EXCLUDE_IDS에 추가합니다. 제외 후 부족한 칸은 새로 촬영해야 합니다.
회차와 손·동작별 최소 40개 표본이 필요합니다. 이는 학습 실행을 위한 최소 조건이며 정확도 보장이 아닙니다.


In [ ]:
EXCLUDE_IDS = ['20260929_151954_73b68e04']  # 오른손 보로 설정했지만 주먹을 촬영한 항목
selected=[(m,r) for m,r in clips if m['id'] not in EXCLUDE_IDS]
coverage={(s,h,l):0 for s in ['train','val','test'] for h in ['left','right'] for l in labels}
data={s:[] for s in ['train','val','test']}
used_ids=set()
for m,rows in selected:
    assert m['id'] not in used_ids, '중복 촬영 ID'
    used_ids.add(m['id'])
    coverage[m['split'],m['hand'],m['label']]+=len(rows)
    data[m['split']].extend((row,labels.index(m['label']),m['hand'],m['id']) for row in rows)
missing=[(k,v) for k,v in coverage.items() if v<40]
assert not missing, f'부족한 회차/손/동작을 촬영하세요: {missing}'
arrays={}
for split,rows in data.items():
    x=np.asarray([r[0]['features'] for r in rows],np.float32)
    y=np.asarray([r[1] for r in rows],np.int32)
    assert x.shape[1]==222 and np.isfinite(x).all()
    arrays[split]=(x,y)
    print(split,x.shape,'클래스 수:',np.bincount(y,minlength=4))
xtrain,ytrain=arrays['train'];xval,yval=arrays['val'];xtest,ytest=arrays['test']


In [ ]:
# 정규화 통계는 학습용 데이터만 사용합니다. 정규화는 TFLite 모델에 포함됩니다.
mean=xtrain.mean(axis=0)
std=np.maximum(xtrain.std(axis=0),0.03)
model=tf.keras.Sequential([
    tf.keras.layers.Input(shape=(222,),name='hand_features'),
    tf.keras.layers.Rescaling(scale=1/std,offset=-mean/std),
    tf.keras.layers.Dense(64,activation='relu',kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.Dropout(0.15),
    tf.keras.layers.Dense(32,activation='relu'),
    tf.keras.layers.Dense(4,activation='softmax',name='rps_probabilities')])
model.compile(optimizer=tf.keras.optimizers.Adam(0.001),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
counts=np.bincount(ytrain,minlength=4)
weights={i:len(ytrain)/(4*int(counts[i])) for i in range(4)}
history=model.fit(xtrain,ytrain,validation_data=(xval,yval),batch_size=32,epochs=150,
    class_weight=weights,callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=15,restore_best_weights=True)],verbose=2)
plt.plot(history.history['loss'],label='train');plt.plot(history.history['val_loss'],label='validation');plt.legend();plt.show()


## TFLite 변환과 독립 평가
판정 보류를 포함한 임계값은 2회차에서만 선택합니다. 3회차는 최종 평가 전용입니다.
아래 성적은 손이 검출된 프레임의 성적입니다. 손 검출 실패는 촬영 시 건너뜀 횟수로 따로 확인합니다.
가위·보의 개선, 기타 오판정, 좌우 손별 결과를 확인하고 현장 테스트 후 교체하세요.


In [ ]:
converter=tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS]
tflite=converter.convert()
Path('rps_custom.tflite').write_bytes(tflite)
interpreter=tf.lite.Interpreter(model_content=tflite,num_threads=1)
interpreter.allocate_tensors()
input_detail=interpreter.get_input_details()[0];output_detail=interpreter.get_output_details()[0]
def lite_predict(x):
    predictions=[]
    for row in x:
        interpreter.set_tensor(input_detail['index'],row[None,:].astype(np.float32))
        interpreter.invoke()
        predictions.append(interpreter.get_tensor(output_detail['index'])[0].copy())
    return np.asarray(predictions)
pv=lite_predict(xval)
assert np.max(np.abs(pv-model.predict(xval,verbose=0)))<0.001,'변환 전후 출력 차이가 큽니다'
def decide(p,threshold,margin=0.1):
    ordered=np.sort(p,axis=1)
    return np.where((ordered[:,-1]>=threshold)&(ordered[:,-1]-ordered[:,-2]>=margin),p.argmax(axis=1),-1)
candidates=np.arange(.50,.951,.05)
scores=[f1_score(yval,decide(pv,t),labels=list(range(4)),average='macro',zero_division=0) for t in candidates]
threshold=float(candidates[int(np.argmax(scores))])
pt=lite_predict(xtest);pred=decide(pt,threshold)
print('선택된 임계값:',threshold)
print(classification_report(ytest,pred,labels=list(range(4)),target_names=labels,zero_division=0))
cm=confusion_matrix(ytest,pred,labels=[0,1,2,3,-1])
print('혼동행렬: 행=실제, 열=예측, 순서 rock/paper/scissors/other/보류')
print(cm)
for hand in ['left','right']:
    mask=np.asarray([r[2]==hand for r in data['test']])
    print(hand,'정답률(보류도 실패로 계산)',float(np.mean(pred[mask]==ytest[mask])))
mapping={'Closed_Fist':0,'Open_Palm':1,'Victory':2}
baseline=np.asarray([mapping.get(r[0]['model_label'],-1) if r[0]['model_score']>=.65 else -1 for r in data['test']])
for i,label in enumerate(labels[:3]):
    mask=ytest==i
    print(label,'기존 기본 모델:',float(np.mean(baseline[mask]==i)),'새 분류기:',float(np.mean(pred[mask]==i)))
# 비교 대상은 원래 Google 분류 결과이며 V2 규칙/V3 개인화 전체와의 비교는 아닙니다.
report={'threshold':threshold,'margin':.1,'test_accuracy_including_rejects':float(np.mean(pred==ytest)),
    'test_coverage':float(np.mean(pred>=0)),'confusion_matrix':cm.tolist(),
    'tensorflow':tf.__version__,'clips':[m for m,r in selected]}
Path('evaluation.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
metadata={'feature_version':'world_distances_angles_v1','feature_count':222,'labels':labels,
    'threshold':threshold,'margin':.1,'sha256':hashlib.sha256(tflite).hexdigest()}
Path('rps_custom.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
with zipfile.ZipFile('rps_trained_model.zip','w',zipfile.ZIP_DEFLATED) as out:
    for name in ['rps_custom.tflite','rps_custom.json','evaluation.json']:
        out.write(name)
print('완료. 평가 결과를 확인한 후 아래 셀로 다운로드하세요. 모델은 아직 Pi에 적용되지 않았습니다.')


In [ ]:
files.download('rps_trained_model.zip')


## 라즈베리파이 적용
다운로드한 파일과 평가 결과를 전달하면 Pi용 경량 실행기에서 입력/출력·연산 호환성을 확인하고 연결합니다.
현재 게임 파일을 덮어쓰거나 models/의 Google 모델을 삭제하지 마세요.
공식 변환 문서: https://www.tensorflow.org/api_docs/python/tf/lite/TFLiteConverter
